In [27]:
import pandas as pd
import numpy as np
import re

In [28]:
df = pd.read_csv('../player_stats_2.csv')

In [29]:
df.head(10)

,player_id,version,name,full_name,description,image,height_cm,weight_kg,dob,positions,...,defending_defensive_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_gk_diving,goalkeeping_gk_handling,goalkeeping_gk_kicking,goalkeeping_gk_positioning,goalkeeping_gk_reflexes,play_styles,url
0,239085,260018,Erling Haaland,Erling Haaland,"Erling Haaland (Erling Braut Håland, born 21 J...",https://cdn.sofifa.net/players/239/085/26_360.png,195.0,94.0,2000-07-21,ST,...,42,47,29,7,14,13,11,7,"Low driven shot, Chip shot, Power shot, Precis...",https://sofifa.com/player/239085/erling-haalan...
1,231747,260018,Kylian Mbappé,Kylian Mbappé,"Kylian Mbappé (Kylian Mbappé Lottin, born 20 D...",https://cdn.sofifa.net/players/231/747/26_360.png,182.0,81.0,1998-12-20,ST,...,26,34,32,13,5,7,11,6,"Quick step, Finesse shot, Acrobatic, Low drive...",https://sofifa.com/player/231747/kylian-mbappe...
2,251854,FC 26,Pedro González López,Pedro González López,Pedri (born 25 November 2002) is a Spanish foo...,https://cdn.sofifa.net/players/251/854/26_360.png,174.0,60.0,2002-11-25,"CM, CDM, CAM",...,74,85,79,12,7,11,8,8,"First touch, Incisive pass, Long ball pass, Ti...",https://sofifa.com/player/251854/pedro-gonzale...
3,255253,FC 26,Vítor Machado Ferreira,Vítor Machado Ferreira,Vitinha (born 13 February 2000) is a Portugues...,https://cdn.sofifa.net/players/255/253/26_360.png,172.0,64.0,2000-02-13,"CM, CDM, CAM",...,74,79,69,12,13,8,5,5,"Technical, Finesse shot, Incisive pass, Long b...",https://sofifa.com/player/255253/vitor-machado...
4,231443,FC 26,Ousmane Dembélé,Ousmane Dembélé,"Ousmane Dembélé (Masour Ousmane Dembélé, born ...",https://cdn.sofifa.net/players/231/443/26_360.png,178.0,67.0,1997-05-15,"ST, RW, CAM",...,49,49,39,6,6,14,10,13,"Rapid, Low driven shot, Pinged pass, Inventive...",https://sofifa.com/player/231443/ousmane-dembe...
5,252371,FC 26,Jude Bellingham,Jude Bellingham,Jude Bellingham (Jude Victor William Bellingha...,https://cdn.sofifa.net/players/252/371/26_360.png,186.0,75.0,2003-06-29,"CAM, CM, LM",...,77,80,82,14,11,10,5,8,"Relentless, Low driven shot, Gamechanger, Inve...",https://sofifa.com/player/252371/jude-bellingh...
6,209331,FC 26,Mohamed Salah,Mohamed Salah,"Mohamed Salah (Mohamed Salah Hamed Ghaly, born...",https://cdn.sofifa.net/players/209/331/26_360.png,175.0,72.0,1992-06-15,"RM, RW",...,38,43,41,14,14,9,11,14,"Finesse shot, Low driven shot, Gamechanger, Wh...",https://sofifa.com/player/209331/mohamed-salah...
7,232580,FC 26,Gabriel dos S. Magalhães,Gabriel dos S. Magalhães,"Gabriel (Gabriel dos Santos Magalhães, born 19...",https://cdn.sofifa.net/players/232/580/26_360.png,190.0,87.0,1997-12-19,CB,...,90,88,90,12,8,13,10,7,"Bruiser, Precision header",https://sofifa.com/player/232580/gabriel-dos-s...
8,277643,FC 26,Lamine Yamal Nasraoui Ebana,Lamine Yamal Nasraoui Ebana,Lamine Yamal (born 13 July 2007) is a Spanish ...,https://cdn.sofifa.net/players/277/643/26_360.png,180.0,72.0,2007-07-13,"RM, RW",...,23,27,28,9,13,7,10,7,"Technical, Finesse shot, Gamechanger, Incisive...",https://sofifa.com/player/277643/lamine-yamal-...
9,235212,FC 26,Achraf Hakimi,Achraf Hakimi,"Achraf Hakimi (Achraf Hakimi Mouh, born 4 Nove...",https://cdn.sofifa.net/players/235/212/26_360.png,181.0,73.0,1998-11-04,"RB, RM",...,82,85,79,10,8,14,6,8,"Relentless, Low driven shot, Whipped pass, Joc...",https://sofifa.com/player/235212/achraf-hakimi...


In [30]:
#Convert all numeric columns to doubles
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[numeric_cols] = df[numeric_cols].astype('float64')

#Convert defending_defensive_awareness, defending_standing_tackle to numeric doubles safely
df['defending_defensive_awareness'] = pd.to_numeric(df['defending_defensive_awareness'], errors='coerce').astype('float64')
df['defending_standing_tackle'] = pd.to_numeric(df['defending_standing_tackle'], errors='coerce').astype('float64')

#convert dob and club_contract_valid_until to datetime
df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
df['club_contract_valid_until'] = pd.to_datetime(df['club_contract_valid_until'], errors='coerce')

#Change preferred_foot to binary
df['preferred_foot'] = df['preferred_foot'].map({'Left': 0, 'Right': 1})

df.head(10)

,player_id,version,name,full_name,description,image,height_cm,weight_kg,dob,positions,...,defending_defensive_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_gk_diving,goalkeeping_gk_handling,goalkeeping_gk_kicking,goalkeeping_gk_positioning,goalkeeping_gk_reflexes,play_styles,url
0,239085.0,260018,Erling Haaland,Erling Haaland,"Erling Haaland (Erling Braut Håland, born 21 J...",https://cdn.sofifa.net/players/239/085/26_360.png,195.0,94.0,2000-07-21,ST,...,42.0,47.0,29.0,7.0,14.0,13.0,11.0,7.0,"Low driven shot, Chip shot, Power shot, Precis...",https://sofifa.com/player/239085/erling-haalan...
1,231747.0,260018,Kylian Mbappé,Kylian Mbappé,"Kylian Mbappé (Kylian Mbappé Lottin, born 20 D...",https://cdn.sofifa.net/players/231/747/26_360.png,182.0,81.0,1998-12-20,ST,...,26.0,34.0,32.0,13.0,5.0,7.0,11.0,6.0,"Quick step, Finesse shot, Acrobatic, Low drive...",https://sofifa.com/player/231747/kylian-mbappe...
2,251854.0,FC 26,Pedro González López,Pedro González López,Pedri (born 25 November 2002) is a Spanish foo...,https://cdn.sofifa.net/players/251/854/26_360.png,174.0,60.0,2002-11-25,"CM, CDM, CAM",...,74.0,85.0,79.0,12.0,7.0,11.0,8.0,8.0,"First touch, Incisive pass, Long ball pass, Ti...",https://sofifa.com/player/251854/pedro-gonzale...
3,255253.0,FC 26,Vítor Machado Ferreira,Vítor Machado Ferreira,Vitinha (born 13 February 2000) is a Portugues...,https://cdn.sofifa.net/players/255/253/26_360.png,172.0,64.0,2000-02-13,"CM, CDM, CAM",...,74.0,79.0,69.0,12.0,13.0,8.0,5.0,5.0,"Technical, Finesse shot, Incisive pass, Long b...",https://sofifa.com/player/255253/vitor-machado...
4,231443.0,FC 26,Ousmane Dembélé,Ousmane Dembélé,"Ousmane Dembélé (Masour Ousmane Dembélé, born ...",https://cdn.sofifa.net/players/231/443/26_360.png,178.0,67.0,1997-05-15,"ST, RW, CAM",...,49.0,49.0,39.0,6.0,6.0,14.0,10.0,13.0,"Rapid, Low driven shot, Pinged pass, Inventive...",https://sofifa.com/player/231443/ousmane-dembe...
5,252371.0,FC 26,Jude Bellingham,Jude Bellingham,Jude Bellingham (Jude Victor William Bellingha...,https://cdn.sofifa.net/players/252/371/26_360.png,186.0,75.0,2003-06-29,"CAM, CM, LM",...,77.0,80.0,82.0,14.0,11.0,10.0,5.0,8.0,"Relentless, Low driven shot, Gamechanger, Inve...",https://sofifa.com/player/252371/jude-bellingh...
6,209331.0,FC 26,Mohamed Salah,Mohamed Salah,"Mohamed Salah (Mohamed Salah Hamed Ghaly, born...",https://cdn.sofifa.net/players/209/331/26_360.png,175.0,72.0,1992-06-15,"RM, RW",...,38.0,43.0,41.0,14.0,14.0,9.0,11.0,14.0,"Finesse shot, Low driven shot, Gamechanger, Wh...",https://sofifa.com/player/209331/mohamed-salah...
7,232580.0,FC 26,Gabriel dos S. Magalhães,Gabriel dos S. Magalhães,"Gabriel (Gabriel dos Santos Magalhães, born 19...",https://cdn.sofifa.net/players/232/580/26_360.png,190.0,87.0,1997-12-19,CB,...,90.0,88.0,90.0,12.0,8.0,13.0,10.0,7.0,"Bruiser, Precision header",https://sofifa.com/player/232580/gabriel-dos-s...
8,277643.0,FC 26,Lamine Yamal Nasraoui Ebana,Lamine Yamal Nasraoui Ebana,Lamine Yamal (born 13 July 2007) is a Spanish ...,https://cdn.sofifa.net/players/277/643/26_360.png,180.0,72.0,2007-07-13,"RM, RW",...,23.0,27.0,28.0,9.0,13.0,7.0,10.0,7.0,"Technical, Finesse shot, Gamechanger, Incisive...",https://sofifa.com/player/277643/lamine-yamal-...
9,235212.0,FC 26,Achraf Hakimi,Achraf Hakimi,"Achraf Hakimi (Achraf Hakimi Mouh, born 4 Nove...",https://cdn.sofifa.net/players/235/212/26_360.png,181.0,73.0,1998-11-04,"RB, RM",...,82.0,85.0,79.0,10.0,8.0,14.0,6.0,8.0,"Relentless, Low driven shot, Whipped pass, Joc...",https://sofifa.com/player/235212/achraf-hakimi...


In [31]:
for col in df.columns:
    print(col, df[col].astype(str).str.contains("sofifa.com").mean())

player_id 0.0
version 0.0
name 0.0
full_name 0.0
description 0.0
image 0.0
height_cm 0.0
weight_kg 0.0
dob 0.0
positions 0.0
overall_rating 0.0
potential 0.0
value 0.0
wage 0.0
preferred_foot 0.0
weak_foot 0.0
skill_moves 0.0
international_reputation 0.0
body_type 0.0
real_face 0.0
release_clause 0.0
specialities 0.0
club_id 0.0
club_name 0.0
club_league_id 0.0
club_league_name 0.0
club_logo 0.0
club_rating 0.0
club_position 0.0
club_kit_number 0.0
club_joined 0.0
club_contract_valid_until 0.0
country_id 0.0
country_name 0.0
country_league_id 0.0
country_league_name 0.0
country_flag 0.0
country_rating 0.0
country_position 0.0
country_kit_number 0.0
attacking_crossing 0.0
attacking_finishing 0.0
attacking_heading_accuracy 0.0
attacking_short_passing 0.0
attacking_volleys 0.0
skill_dribbling 0.0
skill_curve 0.0
skill_fk_accuracy 0.0
skill_long_passing 0.0
skill_ball_control 0.0
movement_acceleration 0.0
movement_sprint_speed 0.0
movement_agility 0.0
movement_reactions 0.0
movement_balanc

In [32]:
cols_to_drop = [
    'description',
    'international_reputation', 
    'club_rating',
    'club_kit_number',
    'club_logo',
    'club_joined',
    'country_id',
    'country_name',
    'country_league_id',
    'country_league_name',
    'country_flag',
    'country_rating',
    'country_position',
    'country_kit_number',
    'mentality_att_positioning',
    'play_styles',
    'url',
    'image',
    'body_type',
    'real_face',
    'specialities',]

df.drop(columns=cols_to_drop, inplace=True)
df.head(10)

,player_id,version,name,full_name,height_cm,weight_kg,dob,positions,overall_rating,potential,...,mentality_penalties,mentality_composure,defending_defensive_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_gk_diving,goalkeeping_gk_handling,goalkeeping_gk_kicking,goalkeeping_gk_positioning,goalkeeping_gk_reflexes
0,239085.0,260018,Erling Haaland,Erling Haaland,195.0,94.0,2000-07-21,ST,91.0,93.0,...,90.0,86.0,42.0,47.0,29.0,7.0,14.0,13.0,11.0,7.0
1,231747.0,260018,Kylian Mbappé,Kylian Mbappé,182.0,81.0,1998-12-20,ST,91.0,92.0,...,84.0,88.0,26.0,34.0,32.0,13.0,5.0,7.0,11.0,6.0
2,251854.0,FC 26,Pedro González López,Pedro González López,174.0,60.0,2002-11-25,"CM, CDM, CAM",90.0,93.0,...,53.0,92.0,74.0,85.0,79.0,12.0,7.0,11.0,8.0,8.0
3,255253.0,FC 26,Vítor Machado Ferreira,Vítor Machado Ferreira,172.0,64.0,2000-02-13,"CM, CDM, CAM",90.0,92.0,...,88.0,88.0,74.0,79.0,69.0,12.0,13.0,8.0,5.0,5.0
4,231443.0,FC 26,Ousmane Dembélé,Ousmane Dembélé,178.0,67.0,1997-05-15,"ST, RW, CAM",90.0,90.0,...,80.0,88.0,49.0,49.0,39.0,6.0,6.0,14.0,10.0,13.0
5,252371.0,FC 26,Jude Bellingham,Jude Bellingham,186.0,75.0,2003-06-29,"CAM, CM, LM",90.0,94.0,...,74.0,90.0,77.0,80.0,82.0,14.0,11.0,10.0,5.0,8.0
6,209331.0,FC 26,Mohamed Salah,Mohamed Salah,175.0,72.0,1992-06-15,"RM, RW",90.0,90.0,...,88.0,91.0,38.0,43.0,41.0,14.0,14.0,9.0,11.0,14.0
7,232580.0,FC 26,Gabriel dos S. Magalhães,Gabriel dos S. Magalhães,190.0,87.0,1997-12-19,CB,89.0,90.0,...,51.0,79.0,90.0,88.0,90.0,12.0,8.0,13.0,10.0,7.0
8,277643.0,FC 26,Lamine Yamal Nasraoui Ebana,Lamine Yamal Nasraoui Ebana,180.0,72.0,2007-07-13,"RM, RW",89.0,95.0,...,75.0,84.0,23.0,27.0,28.0,9.0,13.0,7.0,10.0,7.0
9,235212.0,FC 26,Achraf Hakimi,Achraf Hakimi,181.0,73.0,1998-11-04,"RB, RM",89.0,90.0,...,68.0,84.0,82.0,85.0,79.0,10.0,8.0,14.0,6.0,8.0


In [33]:
for col in df.columns:
    print(col, df[col].astype(str).str.contains("sofifa.com").mean())

player_id 0.0
version 0.0
name 0.0
full_name 0.0
height_cm 0.0
weight_kg 0.0
dob 0.0
positions 0.0
overall_rating 0.0
potential 0.0
value 0.0
wage 0.0
preferred_foot 0.0
weak_foot 0.0
skill_moves 0.0
release_clause 0.0
club_id 0.0
club_name 0.0
club_league_id 0.0
club_league_name 0.0
club_position 0.0
club_contract_valid_until 0.0
attacking_crossing 0.0
attacking_finishing 0.0
attacking_heading_accuracy 0.0
attacking_short_passing 0.0
attacking_volleys 0.0
skill_dribbling 0.0
skill_curve 0.0
skill_fk_accuracy 0.0
skill_long_passing 0.0
skill_ball_control 0.0
movement_acceleration 0.0
movement_sprint_speed 0.0
movement_agility 0.0
movement_reactions 0.0
movement_balance 0.0
power_shot_power 0.0
power_jumping 0.0
power_stamina 0.0
power_strength 0.0
power_long_shots 0.0
mentality_aggression 0.0
mentality_interceptions 0.0
mentality_vision 0.0
mentality_penalties 0.0
mentality_composure 0.0
defending_defensive_awareness 0.0
defending_standing_tackle 0.0
defending_sliding_tackle 0.0
goalke

In [34]:
import numpy as np
import pandas as pd

#cols_to_drop_final = [
   # "defending_defensive_awareness",
   # "defending_standing_tackle",
   # "mentality_attack_position",
#]

# Drop the columns (if present)
#df = df.drop(columns=cols_to_drop_final, errors="ignore")

# If release_clause is missing, set it to 0
if "release_clause" in df.columns:
    df["release_clause"] = pd.to_numeric(df["release_clause"], errors="coerce").fillna(0)

# Fill missing club_name and club_league_name with "Free Agent"
if "club_name" in df.columns:
    df["club_name"] = df["club_name"].fillna("Free Agent")

if "club_league_name" in df.columns:
    df["club_league_name"] = df["club_league_name"].fillna("Free Agent")

# Fill missing club_position with "FREE"
if "club_position" in df.columns:
    df["club_position"] = df["club_position"].fillna("FREE")

# Fill missing club_contract_valid_until with 1900-01-01 to indicate no contract
if "club_contract_valid_until" in df.columns:
    # Convert to datetime first, then fill missing with sentinel date
    df["club_contract_valid_until"] = pd.to_datetime(df["club_contract_valid_until"], errors="coerce")
    df["club_contract_valid_until"] = df["club_contract_valid_until"].fillna(pd.Timestamp("1900-01-01"))

    # (Recommended) explicit flag so the model can learn “no contract” directly
    df["no_contract"] = (df["club_contract_valid_until"] == pd.Timestamp("1900-01-01")).astype(int)


# Create binary free agent column
if "club_name" in df.columns and "club_league_name" in df.columns:
    df["free_agent"] = np.where(
        (df["club_name"] == "Free Agent") & (df["club_league_name"] == "Free Agent"),
        1,
        0,
    )

df.head(10)

,player_id,version,name,full_name,height_cm,weight_kg,dob,positions,overall_rating,potential,...,defending_defensive_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_gk_diving,goalkeeping_gk_handling,goalkeeping_gk_kicking,goalkeeping_gk_positioning,goalkeeping_gk_reflexes,no_contract,free_agent
0,239085.0,260018,Erling Haaland,Erling Haaland,195.0,94.0,2000-07-21,ST,91.0,93.0,...,42.0,47.0,29.0,7.0,14.0,13.0,11.0,7.0,0,0
1,231747.0,260018,Kylian Mbappé,Kylian Mbappé,182.0,81.0,1998-12-20,ST,91.0,92.0,...,26.0,34.0,32.0,13.0,5.0,7.0,11.0,6.0,0,0
2,251854.0,FC 26,Pedro González López,Pedro González López,174.0,60.0,2002-11-25,"CM, CDM, CAM",90.0,93.0,...,74.0,85.0,79.0,12.0,7.0,11.0,8.0,8.0,0,0
3,255253.0,FC 26,Vítor Machado Ferreira,Vítor Machado Ferreira,172.0,64.0,2000-02-13,"CM, CDM, CAM",90.0,92.0,...,74.0,79.0,69.0,12.0,13.0,8.0,5.0,5.0,0,0
4,231443.0,FC 26,Ousmane Dembélé,Ousmane Dembélé,178.0,67.0,1997-05-15,"ST, RW, CAM",90.0,90.0,...,49.0,49.0,39.0,6.0,6.0,14.0,10.0,13.0,0,0
5,252371.0,FC 26,Jude Bellingham,Jude Bellingham,186.0,75.0,2003-06-29,"CAM, CM, LM",90.0,94.0,...,77.0,80.0,82.0,14.0,11.0,10.0,5.0,8.0,0,0
6,209331.0,FC 26,Mohamed Salah,Mohamed Salah,175.0,72.0,1992-06-15,"RM, RW",90.0,90.0,...,38.0,43.0,41.0,14.0,14.0,9.0,11.0,14.0,0,0
7,232580.0,FC 26,Gabriel dos S. Magalhães,Gabriel dos S. Magalhães,190.0,87.0,1997-12-19,CB,89.0,90.0,...,90.0,88.0,90.0,12.0,8.0,13.0,10.0,7.0,0,0
8,277643.0,FC 26,Lamine Yamal Nasraoui Ebana,Lamine Yamal Nasraoui Ebana,180.0,72.0,2007-07-13,"RM, RW",89.0,95.0,...,23.0,27.0,28.0,9.0,13.0,7.0,10.0,7.0,0,0
9,235212.0,FC 26,Achraf Hakimi,Achraf Hakimi,181.0,73.0,1998-11-04,"RB, RM",89.0,90.0,...,82.0,85.0,79.0,10.0,8.0,14.0,6.0,8.0,0,0


In [35]:
# ================================
# Deduplication + GK mean-impute
# (drop-in cells for your script)
# ================================

import numpy as np
import pandas as pd


# ----------------
# 1) DEDUPLICATION
# ----------------
# Goal: keep ONE row per player_id.
# Rule: keep the row with the highest 'version' (latest).
# Fallback (if version missing): keep first occurrence.

def dedupe_player_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Deduplicate dataset based only on player_id.

    Keeps the first occurrence of each player_id and removes any duplicates.
    """

    if "player_id" not in df.columns:
        raise ValueError("player_id column is required for deduplication.")

    # Ensure player_id is numeric and consistent
    df["player_id"] = pd.to_numeric(df["player_id"], errors="coerce").astype("Int64")

    # Drop rows where player_id could not be parsed
    df = df.dropna(subset=["player_id"])

    before = len(df)

    # Remove duplicates based only on player_id
    df = df.drop_duplicates(subset=["player_id"], keep="first").copy()

    after = len(df)

    print(f"Deduplication: {before:,} → {after:,} rows (removed {before-after:,})")

    return df


# Apply
df = dedupe_player_id(df)

Deduplication: 17,499 → 17,499 rows (removed 0)


In [36]:
df.head(100)

,player_id,version,name,full_name,height_cm,weight_kg,dob,positions,overall_rating,potential,...,defending_defensive_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_gk_diving,goalkeeping_gk_handling,goalkeeping_gk_kicking,goalkeeping_gk_positioning,goalkeeping_gk_reflexes,no_contract,free_agent
0,239085,260018,Erling Haaland,Erling Haaland,195.0,94.0,2000-07-21,ST,91.0,93.0,...,42.0,47.0,29.0,7.0,14.0,13.0,11.0,7.0,0,0
1,231747,260018,Kylian Mbappé,Kylian Mbappé,182.0,81.0,1998-12-20,ST,91.0,92.0,...,26.0,34.0,32.0,13.0,5.0,7.0,11.0,6.0,0,0
2,251854,FC 26,Pedro González López,Pedro González López,174.0,60.0,2002-11-25,"CM, CDM, CAM",90.0,93.0,...,74.0,85.0,79.0,12.0,7.0,11.0,8.0,8.0,0,0
3,255253,FC 26,Vítor Machado Ferreira,Vítor Machado Ferreira,172.0,64.0,2000-02-13,"CM, CDM, CAM",90.0,92.0,...,74.0,79.0,69.0,12.0,13.0,8.0,5.0,5.0,0,0
4,231443,FC 26,Ousmane Dembélé,Ousmane Dembélé,178.0,67.0,1997-05-15,"ST, RW, CAM",90.0,90.0,...,49.0,49.0,39.0,6.0,6.0,14.0,10.0,13.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,264652,FC 26,Bradley Barcola,Bradley Barcola,188.0,70.0,2002-09-02,"LW, RW, LM",84.0,88.0,...,34.0,39.0,28.0,8.0,8.0,15.0,6.0,6.0,0,0
96,20801,FC 26,C. Ronaldo dos Santos Aveiro,C. Ronaldo dos Santos Aveiro,187.0,85.0,1985-02-05,ST,85.0,85.0,...,24.0,32.0,24.0,7.0,11.0,15.0,14.0,11.0,0,0
97,165153,FC 26,Karim Benzema,Karim Benzema,185.0,81.0,1987-12-19,"ST, CAM",85.0,85.0,...,43.0,24.0,18.0,13.0,11.0,5.0,5.0,7.0,0,0
98,257279,FC 26,Alejandro Baena Rodríguez,Alejandro Baena Rodríguez,175.0,70.0,2001-07-20,"LM, LW",84.0,88.0,...,67.0,66.0,64.0,6.0,11.0,8.0,12.0,11.0,0,0


In [37]:
df.to_csv('../player_stats_cleaned.csv', index=False)

In [38]:
import pandas as pd

df_2 = pd.read_csv("../player_stats_cleaned.csv", low_memory=False)

print("shape:", df_2.shape)
print("version non-null:", df_2["version"].notna().sum())
print("unique player_id:", df_2["player_id"].nunique(), "duplicates:", df_2["player_id"].duplicated().sum())

# Misalignment signal check
print("any http anywhere:", df_2.astype(str).apply(lambda c: c.str.contains("http", na=False).any()).any())

shape: (17499, 57)
version non-null: 17499
unique player_id: 17499 duplicates: 0
any http anywhere: False
